In [1]:
import torch
import data_manager as dm
from unet_better_model import get_model
import os
from diffusers import DDPMScheduler
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
import json
import torch
from variational_autoencoder import VariationalAutoencoder

/opt/conda/envs/dl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
noise_scheduler = DDPMScheduler(
        num_train_timesteps=1000,
        beta_start=1e-4,
        beta_end=0.02,
        beta_schedule="linear",
        clip_sample=False
    )
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DiffusionModel:
    def __init__(self, exp_path, noise_scheduler, device):
        with open(os.path.join(exp_path, 'config.txt'), 'r') as file:
            lines = file.readlines()
            model_name = [x for x in lines if x.startswith('Model name:')][0].rstrip().split()[-1]
        self.model = get_model(model_name)()
        model_path = os.path.join(exp_path, 'final_model.pth')
        self.model.load_state_dict(torch.load(model_path, map_location='cpu'))
        self.model = self.model.to(device)
        self.noise_scheduler = noise_scheduler
        self.device = device
        self.img_size = 32
    
    def generate_noise(self, seed=42, num_samples=1):
        torch.manual_seed(seed)
        x = torch.randn(num_samples, 3, 32, 32).to(self.device)
        return x
    
    def generate_image(self, noise):
        noise_scheduler = self.noise_scheduler
        device = self.device
        model = self.model
        
        def denormalize(x):
            return (x + 1) / 2
        
        model.eval()
        with torch.no_grad():
            x = noise
            for step in noise_scheduler.timesteps:
                t = torch.tensor([step], device=device).expand(x.size(0))
                pred_noise = model(x, t)
                x = noise_scheduler.step(pred_noise, step, x).prev_sample
            x = denormalize(x).clamp(0, 1)
        model.train()
        return x

In [3]:
def create_vae_from_config(config_path, device='cpu'):
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = VariationalAutoencoder(
        img_size=config.get('img_size', 64),
        emb_dimension=config.get('emb_dim', 2),
        device=device,
        in_channels=config.get('in_channels', 3),
        base_channels=config.get('base_channels', 128),
        num_blocks=config.get('num_blocks', 4),
        kernel_size=config.get('kernel_size', 2),
        stride=config.get('stride', 2)
    )
    
    return model.to(device)

class VAE_Model:
    def __init__(self, exp_path, device):
        config_path = os.path.join(exp_path, 'config.json')
        self.device = device
        self.model = create_vae_from_config(config_path, device=self.device)
        model_path = os.path.join(exp_path, 'final_model.pth')
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.eval() 
        self.exp_path = exp_path
        self.emb_dim = self.model.emb_dimension
        self.img_size = 64
    
    def generate_noise(self, num_samples=1, mean=0.0, std=1.0, seed=42):
        torch.manual_seed(seed)
        return torch.randn(num_samples, self.emb_dim, device=self.device) * std + mean
    
    def generate_image(self, noise=None, num_samples=1):
        self.model.eval()
        with torch.no_grad():
            if noise is None:
                noise = self.generate_noise(num_samples=num_samples)
            noise = noise.to(self.device)
            generated = self.model.decoder(noise)
            if generated.min() < -0.5:
                generated = torch.clamp(generated, -1, 1)
            else:
                generated = torch.clamp(generated, 0, 1)
        return generated

In [4]:
exp_path_diff = os.path.join('experiments_diffusion', 'exp_20250609_103308')

In [5]:
exp_path_vae = os.path.join('experiments_vae', 'exp_20250609_133233')

In [6]:
model = DiffusionModel(exp_path_diff, noise_scheduler, device)

In [7]:
model = VAE_Model(exp_path_vae, device)

In [8]:
def interpolate(model, seed_1, seed_2, save_path="interpolation.png"):
    noise1 = model.generate_noise(seed=seed_1)
    noise2 = model.generate_noise(seed=seed_2)

    def interpolate_noise(n1, n2, steps=8):
        return [(1 - alpha) * n1 + alpha * n2 for alpha in torch.linspace(0, 1, steps + 2)]

    interpolated_noises = interpolate_noise(noise1, noise2, steps=8)

    fig, axs = plt.subplots(1, 10, figsize=(20, 3))

    for i, noise in tqdm(enumerate(interpolated_noises)):
        with torch.no_grad():
            img = model.generate_image(noise).squeeze(0)  # shape: [3, 32, 32]

        image = TF.to_pil_image(img)
        axs[i].imshow(image)
        axs[i].axis("off")
        axs[i].set_title(f"Step {i}")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)

In [13]:
# VAE
interpolate(model, 10, 42, 'vae_interpolation_1.png')
interpolate(model, 123, 65, 'vae_interpolation_2.png')
interpolate(model, 987, 916, 'vae_interpolation_3.png')

10it [00:00, 101.28it/s]


In [41]:
# Diffusion
interpolate(model, 10, 42, 'diffusion_interpolation_1.png')
interpolate(model, 123, 87, 'diffusion_interpolation_2.png')
interpolate(model, 9, 12, 'diffusion_interpolation_3.png')

10it [01:57, 11.74s/it]
10it [01:56, 11.61s/it]
10it [02:01, 12.11s/it]
